In [ ]:
import numpy as np
import pandas as pd

benchmark = pd.read_csv("")
pb_data = pd.read_csv("")

In [ ]:
import numpy as np
import pandas as pd

def to_float(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().replace(',', '').replace('−','-')
    try:
        return float(s)
    except:
        return np.nan

def to_int01(x):
    v = to_float(x)
    return np.nan if pd.isna(v) else int(v)

def clean_str(x):
    if pd.isna(x): return np.nan
    s = str(x).strip()
    return ' '.join(s.split())

def normalize_tolerance(val):
    if pd.isna(val): return np.nan
    s = (str(val).strip().lower()
                   .replace('−','-')
                   .replace(' ','')
                   .replace(',',''))
    mapping = {
        '1e-06':1e-6, '1e-6':1e-6, '0.000001':1e-6,
        '1e-05':1e-5, '1e-5':1e-5, '0.00001':1e-5,
        '1e-04':1e-4, '1e-4':1e-4, '0.0001':1e-4,
        '0':'nan', '0.0':'nan'
    }
    s = mapping.get(s, s)
    try:
        return float(s)
    except:
        return np.nan

def normalize_to_schema(df, source, tol_order=(1e-6,1e-5,1e-4)):
    t = df.copy()

    # 字符列
    t["Split"]    = t["Split"].map(clean_str)
    t["MolName"]  = t["Mol Name"].map(clean_str)
    if "Solver Name" in t.columns:
        t["Solver"] = t["Solver Name"].map(clean_str)
    else:
        t["Solver"] = np.nan

    # 数值列
    t["AtomNum"]  = t["Atom Number"].apply(to_float)
    t["Grid"]     = t["Grid Size"].apply(to_float)
    t["Tol_num"]  = t["Tolerance"].map(normalize_tolerance)
    t["CUDA"]     = t["CUDA Used"].apply(to_int01)           if "CUDA Used" in t.columns else np.nan
    t["Succeeded"]= t["Succeeded"].apply(to_int01)           if "Succeeded" in t.columns else np.nan
    t["EPB"]      = t["EPB Value"].apply(to_float)           if "EPB Value" in t.columns else np.nan
    t["Time"]     = t["Elapsed Time"].apply(to_float)        if "Elapsed Time" in t.columns else np.nan

    t["Tol_cat"] = pd.Categorical(t["Tol_num"], categories=list(tol_order), ordered=True)

    cols = ["Split","MolName","AtomNum","Grid","Tol_num","Tol_cat","CUDA","Succeeded","Solver","EPB","Time"]
    t = t[cols].copy()

    t = t.dropna(subset=["Split","MolName","AtomNum","Grid","Tol_num"], how="any")

    t["source"] = source
    return t

bench_norm = normalize_to_schema(benchmark, source="bench")
data_norm  = normalize_to_schema(pb_data,   source="data")

print("bench Tol:", sorted(bench_norm["Tol_num"].dropna().unique()))
print("data  Tol:", sorted(data_norm["Tol_num"].dropna().unique()))
print(bench_norm.dtypes[["Split","MolName","AtomNum","Grid","Tol_num","EPB","Time"]])
print(data_norm.dtypes[ ["Split","MolName","AtomNum","Grid","Tol_num","EPB","Time"]])


In [ ]:
merge_keys = ["Split","MolName","AtomNum","Grid","Tol_num","CUDA"]
df = pd.merge(
    bench_norm.add_suffix("_Bench"),
    data_norm.add_suffix("_Data"),
    left_on=[k+"_Bench" for k in merge_keys],
    right_on=[k+"_Data"  for k in merge_keys],
    how="inner"
)

df["EPB_abs_diff"]      = (df["EPB_Data"] - df["EPB_Bench"]).abs()
df["EPB_abs_diff_norm"] = df["EPB_abs_diff"] / df["AtomNum_Bench"]

out = df[[
    "Split_Bench","MolName_Bench","AtomNum_Bench","Grid_Bench","Tol_num_Bench","Tol_cat_Bench","CUDA_Bench",
    "Solver_Data","EPB_Bench","EPB_Data","Time_Bench","Time_Data","Succeeded_Bench","Succeeded_Data",
    "EPB_abs_diff","EPB_abs_diff_norm"
]].rename(columns={
    "Split_Bench":"Split",
    "MolName_Bench":"Mol Name",
    "AtomNum_Bench":"Atom Number",
    "Grid_Bench":"Grid Size",
    "Tol_num_Bench":"Tolerance",
    "Tol_cat_Bench":"Tolerance_cat",
    "CUDA_Bench":"CUDA Used"
})

print("Merged shape:", out.shape)
print(out.head())
